# 02. Autograd (자동 미분)

PyTorch는 텐서 연산을 기록해 계산 그래프를 만들고, `backward()` 한 번으로 모든 기울기를 계산합니다.

**학습 목표**
- `requires_grad`와 `backward()`로 기울기를 구한다
- 계산 그래프와 기울기 누적, `no_grad`를 이해한다
- autograd만으로 경사하강법을 구현한다

In [ ]:
import torch

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch {torch.__version__}, device: {device}")

## 1. 기본 사용법

$y = x^2 + 3x$ 일 때 $\frac{dy}{dx} = 2x + 3$ 이므로 $x=2$에서 기울기는 7입니다.

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = x ** 2 + 3 * x
y.backward()
print("y =", y.item(), "| dy/dx =", x.grad.item())

## 2. 계산 그래프

각 결과 텐서는 자신을 만든 연산을 `grad_fn`으로 기억합니다.

In [ ]:
a = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
b = a * 2
c = b.sum()
print("b.grad_fn:", b.grad_fn)
print("c.grad_fn:", c.grad_fn)
c.backward()
print("dc/da:", a.grad)   # 각 원소에 대해 2

## 3. 기울기 누적

`backward()`를 여러 번 호출하면 `.grad`에 **더해집니다**. 그래서 학습 루프에서는 매 스텝마다 기울기를 0으로 초기화합니다.

In [ ]:
w = torch.tensor(1.0, requires_grad=True)
for i in range(3):
    loss = 3 * w
    loss.backward()
    print(f"step {i}: w.grad = {w.grad.item()}")

w.grad.zero_()
print("초기화 후:", w.grad.item())

## 4. 기울기 추적 끄기

추론이나 파라미터 직접 갱신처럼 그래프가 필요 없을 때는 `torch.no_grad()` 또는 `.detach()`를 사용합니다.

In [ ]:
x = torch.tensor(1.0, requires_grad=True)
with torch.no_grad():
    y = x * 2
print("no_grad 안 결과 requires_grad:", y.requires_grad)
print("detach 결과 requires_grad   :", (x * 2).detach().requires_grad)

## 5. 벡터 함수의 기울기

출력이 스칼라가 아니면 `backward(gradient=...)`로 가중치를 넘깁니다. 보통은 손실을 스칼라로 만들어 사용합니다.

In [ ]:
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y = x ** 3
y.backward(gradient=torch.ones_like(y))
print("dy/dx = 3x^2:", x.grad)

## 6. autograd로 경사하강법 구현

$y = 2x + 1$ 에 노이즈를 더한 데이터에서 `w`, `b`를 직접 갱신해 찾아봅니다. 데이터는 노트북 안에서 생성하므로 `data/` 폴더는 사용하지 않습니다.

In [ ]:
X = torch.linspace(-1, 1, 100)
Y = 2 * X + 1 + 0.1 * torch.randn(100)

w = torch.zeros(1, requires_grad=True)
b = torch.zeros(1, requires_grad=True)
lr = 0.1
losses = []

for epoch in range(100):
    pred = w * X + b
    loss = ((pred - Y) ** 2).mean()
    loss.backward()
    with torch.no_grad():
        w -= lr * w.grad
        b -= lr * b.grad
        w.grad.zero_()
        b.grad.zero_()
    losses.append(loss.item())
    if epoch % 20 == 0:
        print(f"epoch {epoch:3d} | loss {loss.item():.4f} | w {w.item():.3f} | b {b.item():.3f}")

print(f"최종: w = {w.item():.3f}, b = {b.item():.3f} (정답 w=2, b=1)")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(losses)
axes[0].set(title="Loss", xlabel="epoch", ylabel="MSE")
axes[1].scatter(X, Y, s=8, alpha=0.6, label="data")
axes[1].plot(X, (w * X + b).detach(), color="red", label="fit")
axes[1].set(title="Fit result")
axes[1].legend()
plt.tight_layout()
plt.show()

## 요약
- `requires_grad=True`인 텐서의 연산은 그래프로 기록되고 `backward()`로 기울기가 계산된다
- 기울기는 누적되므로 매 스텝 `zero_()`(또는 `optimizer.zero_grad()`)가 필요하다
- 파라미터 갱신과 추론은 `torch.no_grad()` 안에서 한다
- 다음 섹션에서는 이 과정을 `nn.Module`과 `optim`으로 간결하게 만든다